# Phase 3d: Model Evaluation and Selection
## DNA Gene Mapping Project
**Author:** Sharique Mohammad  
**Date:** February 2026  

---

## Objective
Final model evaluation and selection:
- Compare all models (baseline + ensemble)
- Select best model for each task
- Evaluate on test set (final performance)
- Generate comprehensive reports

## Deliverables
- Best model selection with justification
- Test set performance (unbiased evaluation)
- ROC/PR curves comparison
- Final model recommendations

---
## 1. Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, accuracy_score,
    average_precision_score
)

import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("OK Imports successful")

In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "ml"
MODEL_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "data" / "ml" / "metrics"
FIGURES_DIR = PROJECT_ROOT / "data" / "analytical" / "figures" / "phase3"

print("="*80)
print("PHASE 3D: MODEL EVALUATION AND SELECTION")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

---
## 2. Load Test Datasets

In [ ]:
print("Loading test datasets...")

# Variant pathogenicity test set
with open(DATA_DIR / "variant_test.pkl", 'rb') as f:
    test_data = pickle.load(f)
    X_test, y_test = test_data['X'], test_data['y']

print(f"Variant test set: {X_test.shape}")
print(f"  Class 0 (Benign): {(y_test == False).sum():,}")
print(f"  Class 1 (Pathogenic): {(y_test == True).sum():,}")

# SV test set
with open(DATA_DIR / "sv_test.pkl", 'rb') as f:
    sv_test = pickle.load(f)
    X_sv_test, y_sv_test = sv_test['X'], sv_test['y']

print(f"\nSV test set: {X_sv_test.shape}")
print(f"  Class 0 (Low-risk): {(y_sv_test == False).sum():,}")
print(f"  Class 1 (High-risk): {(y_sv_test == True).sum():,}")

---
## 3. Load All Trained Models

In [ ]:
print("\nLoading trained models...")

# Variant pathogenicity models
variant_models = {}

# Baseline
with open(MODEL_DIR / "baseline_lr_variants.pkl", 'rb') as f:
    variant_models['Baseline LR'] = pickle.load(f)

with open(MODEL_DIR / "baseline_dt_variants.pkl", 'rb') as f:
    variant_models['Baseline DT'] = pickle.load(f)

# Ensemble
with open(MODEL_DIR / "ensemble_rf_variants.pkl", 'rb') as f:
    variant_models['Random Forest'] = pickle.load(f)

with open(MODEL_DIR / "ensemble_xgb_variants.pkl", 'rb') as f:
    variant_models['XGBoost'] = pickle.load(f)

with open(MODEL_DIR / "ensemble_lgb_variants.pkl", 'rb') as f:
    variant_models['LightGBM'] = pickle.load(f)

print(f"Loaded {len(variant_models)} variant models")

# SV models
sv_models = {}

# Baseline
with open(MODEL_DIR / "baseline_lr_sv.pkl", 'rb') as f:
    sv_models['Baseline LR'] = pickle.load(f)

with open(MODEL_DIR / "baseline_dt_sv.pkl", 'rb') as f:
    sv_models['Baseline DT'] = pickle.load(f)

# Ensemble
with open(MODEL_DIR / "ensemble_rf_sv.pkl", 'rb') as f:
    sv_models['Random Forest'] = pickle.load(f)

with open(MODEL_DIR / "ensemble_xgb_sv.pkl", 'rb') as f:
    sv_models['XGBoost'] = pickle.load(f)

with open(MODEL_DIR / "ensemble_lgb_sv.pkl", 'rb') as f:
    sv_models['LightGBM'] = pickle.load(f)

print(f"Loaded {len(sv_models)} SV models")

---
## 4. Variant Pathogenicity - Test Set Evaluation

In [ ]:
print("\n" + "="*80)
print("VARIANT PATHOGENICITY - TEST SET EVALUATION")
print("="*80)

variant_results = []

for name, model in variant_models.items():
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    results = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'PR-AUC': average_precision_score(y_test, y_proba)
    }
    variant_results.append(results)
    
    print(f"\n{name}:")
    print(f"  F1:        {results['F1']:.4f}")
    print(f"  Precision: {results['Precision']:.4f}")
    print(f"  Recall:    {results['Recall']:.4f}")
    print(f"  ROC-AUC:   {results['ROC-AUC']:.4f}")

# Create comparison DataFrame
variant_comparison = pd.DataFrame(variant_results)
variant_comparison = variant_comparison.sort_values('F1', ascending=False)

print("\n" + "="*80)
print("VARIANT MODELS RANKED BY F1-SCORE")
print("="*80)
print(variant_comparison.to_string(index=False))

# Save results
variant_comparison.to_csv(METRICS_DIR / "variant_test_results.csv", index=False)
print("\nOK Results saved: variant_test_results.csv")

---
## 5. Variant Pathogenicity - ROC Curve Comparison

In [ ]:
print("\nGenerating ROC curve comparison...")

plt.figure(figsize=(10, 8))

for name, model in variant_models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Variant Pathogenicity - ROC Curves (Test Set)', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "25_variant_roc_comparison.png", dpi=150, bbox_inches='tight')
plt.close()

print("OK ROC curve saved")

---
## 6. Variant Pathogenicity - Precision-Recall Curve

In [ ]:
print("Generating PR curve comparison...")

plt.figure(figsize=(10, 8))

for name, model in variant_models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    
    plt.plot(recall, precision, label=f'{name} (AP={pr_auc:.3f})', linewidth=2)

# Baseline (random classifier)
baseline_precision = (y_test == True).sum() / len(y_test)
plt.axhline(y=baseline_precision, color='k', linestyle='--', label=f'Random (AP={baseline_precision:.3f})', linewidth=1)

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Variant Pathogenicity - Precision-Recall Curves (Test Set)', fontsize=14)
plt.legend(loc='lower left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "26_variant_pr_comparison.png", dpi=150, bbox_inches='tight')
plt.close()

print("OK PR curve saved")

---
## 7. Select Best Variant Model

In [ ]:
print("\n" + "="*80)
print("BEST VARIANT MODEL SELECTION")
print("="*80)

# Select by F1 score
best_variant = variant_comparison.iloc[0]
best_variant_name = best_variant['Model']
best_variant_model = variant_models[best_variant_name]

print(f"\nSelected: {best_variant_name}")
print(f"\nTest Set Performance:")
print(f"  F1-Score:  {best_variant['F1']:.4f}")
print(f"  Precision: {best_variant['Precision']:.4f}")
print(f"  Recall:    {best_variant['Recall']:.4f}")
print(f"  ROC-AUC:   {best_variant['ROC-AUC']:.4f}")
print(f"  PR-AUC:    {best_variant['PR-AUC']:.4f}")

# Detailed classification report
y_pred = best_variant_model.predict(X_test)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Pathogenic']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(f"  TN: {cm[0,0]:,}  FP: {cm[0,1]:,}")
print(f"  FN: {cm[1,0]:,}  TP: {cm[1,1]:,}")

# Save best model info
best_variant_info = {
    'model_name': best_variant_name,
    'test_metrics': best_variant.to_dict(),
    'confusion_matrix': cm.tolist(),
    'classification_report': classification_report(y_test, y_pred, target_names=['Benign', 'Pathogenic'], output_dict=True)
}

with open(METRICS_DIR / "best_variant_model.json", 'w') as f:
    json.dump(best_variant_info, f, indent=2)

print("\nOK Best model info saved: best_variant_model.json")

---
## 8. SV Risk - Test Set Evaluation

In [ ]:
print("\n" + "="*80)
print("SV RISK - TEST SET EVALUATION")
print("="*80)

sv_results = []

for name, model in sv_models.items():
    # Predictions
    y_pred = model.predict(X_sv_test)
    y_proba = model.predict_proba(X_sv_test)[:, 1]
    
    # Metrics
    results = {
        'Model': name,
        'Accuracy': accuracy_score(y_sv_test, y_pred),
        'Precision': precision_score(y_sv_test, y_pred),
        'Recall': recall_score(y_sv_test, y_pred),
        'F1': f1_score(y_sv_test, y_pred),
        'ROC-AUC': roc_auc_score(y_sv_test, y_proba),
        'PR-AUC': average_precision_score(y_sv_test, y_proba)
    }
    sv_results.append(results)
    
    print(f"\n{name}:")
    print(f"  F1:        {results['F1']:.4f}")
    print(f"  Precision: {results['Precision']:.4f}")
    print(f"  Recall:    {results['Recall']:.4f}")
    print(f"  ROC-AUC:   {results['ROC-AUC']:.4f}")

# Create comparison DataFrame
sv_comparison = pd.DataFrame(sv_results)
sv_comparison = sv_comparison.sort_values('Recall', ascending=False)

print("\n" + "="*80)
print("SV MODELS RANKED BY RECALL")
print("="*80)
print(sv_comparison.to_string(index=False))

# Check for perfect scores
perfect_models = sv_comparison[sv_comparison['F1'] >= 0.999]
if len(perfect_models) > 0:
    print("\nWARNING: Multiple models achieve near-perfect scores.")
    print("This strongly suggests data leakage in SV features.")
    print("Recommendation: Investigate SV feature engineering.")

# Save results
sv_comparison.to_csv(METRICS_DIR / "sv_test_results.csv", index=False)
print("\nOK Results saved: sv_test_results.csv")

---
## 9. Final Summary Report

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY - PHASE 3 COMPLETE")
print("="*80)

summary = {
    'timestamp': datetime.now().isoformat(),
    'variant_pathogenicity': {
        'best_model': best_variant_name,
        'test_performance': {
            'f1': float(best_variant['F1']),
            'precision': float(best_variant['Precision']),
            'recall': float(best_variant['Recall']),
            'roc_auc': float(best_variant['ROC-AUC']),
            'pr_auc': float(best_variant['PR-AUC'])
        },
        'all_models': variant_results
    },
    'sv_risk': {
        'all_models': sv_results,
        'data_quality_warning': len(perfect_models) > 0
    },
    'datasets': {
        'variant_test_size': len(y_test),
        'sv_test_size': len(y_sv_test)
    }
}

# Save summary
summary_file = METRICS_DIR / "phase3_final_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print("\nVARIANT PATHOGENICITY PREDICTION:")
print(f"  Best Model: {best_variant_name}")
print(f"  Test F1: {best_variant['F1']:.4f}")
print(f"  Test Precision: {best_variant['Precision']:.4f}")
print(f"  Test Recall: {best_variant['Recall']:.4f}")
print(f"  ROC-AUC: {best_variant['ROC-AUC']:.4f}")

print("\nSV RISK PREDICTION:")
if len(perfect_models) > 0:
    print("  Status: Data leakage detected")
    print("  Recommendation: Investigate feature engineering")
else:
    best_sv = sv_comparison.iloc[0]
    print(f"  Best Model: {best_sv['Model']}")
    print(f"  Test Recall: {best_sv['Recall']:.4f}")

print(f"\nOK Final summary saved: {summary_file.name}")

print("\n" + "="*80)
print("FILES CREATED")
print("="*80)
print("\nMetrics:")
print("  - variant_test_results.csv")
print("  - sv_test_results.csv")
print("  - best_variant_model.json")
print("  - phase3_final_summary.json")
print("\nFigures:")
print("  - 25_variant_roc_comparison.png")
print("  - 26_variant_pr_comparison.png")

print("\n" + "="*80)
print("PHASE 3D COMPLETE: MODEL EVALUATION FINISHED")
print("="*80)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nRECOMMENDATIONS:")
print("  1. Use best variant model for deployment")
if len(perfect_models) > 0:
    print("  2. Investigate SV feature leakage before using SV models")
    print("  3. Run SV leakage investigation notebook")
print("\nNext: Phase 3e (Model Explainability with SHAP)")
print("="*80)